# Diseño, implementación y validación experimental de una torreta autónoma basada en aprendizaje por refuerzo profundo y visión artificial (CV) (3 de 3)
- Este es el **tercero** de 3 notebooks donde junto los dos modelos de IA creados (RL para control y CNN para vision) aplicando la detección del objetivo para pasarlo al modelo de RL y traducirlo en comandos a la torrera fisica. Este notebook sería la validación experimental. 
- Los otros dos notebooks se pueden encontrar en la entrega o los enlaces a Kaggle:

1. [parte1-rl-gymnasium](https://www.kaggle.com/code/jorgesu040/parte1-rl-gymnasium)
2. [parte2-cnn-visionartificial](https://www.kaggle.com/code/jorgesu040/parte2-cnn-visionartificial)

> **Nota**: este notebook no se puede ejecutar en Kaggle pues lanza ventanas en el PC

> Al continuación incluyo un links a youtube con videos de la torreta y el sistema en acción:
> - Disparo 1: https://youtu.be/mGippGd279k
> - Disparo 2: https://youtu.be/KS9mdomQKo0
> - Disparo 3: https://youtu.be/VWXIKj431Yc

> Los videos muestran primero la perspectiva desde la webcam y el codigo (la torreta) y luego desde fuera con el móvil

Versión para GitHub de la entrega final. Las rutas apuntan a los archivos incluidos. Puedes empezar por la prueba de imagen sin conectar el Arduino.

# 1. Deployment: Torreta Autónoma (Vision + RL + Serial)
Este notebook integra los tres pilares del proyecto:
1.  **Ojos (Vision):** Detección de dianas con YOLOv8.
2.  **Cerebro (RL):** Modelo DQN entrenado para decidir acciones.
3.  **Cuerpo (Arduino):** Control de servos vía Serial. 

### Requisitos
- Webcam conectada.
- Arduino por USB con [turret.ino](../firmware/turret/turret.ino) para las pruebas de hardware.
- El repositorio incluye el [checkpoint DQN](../models/dqn-policy.zip) y los [pesos YOLO](../models/best.pt).

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

In [ ]:
import cv2
import serial
import time
import numpy as np
from ultralytics import YOLO
from stable_baselines3 import DQN

In [ ]:
# Config del variables

# Config del puerto serial
SERIAL_PORT = None  # Ejemplo con hardware: "COM7" o "/dev/ttyUSB0"
CAMERA_INDEX = 0
BAUD_RATE = 9600

# Rutas de modelos
MODEL_RL_PATH = PROJECT_ROOT / "models" / "dqn-policy.zip"
MODEL_YOLO_PATH = PROJECT_ROOT / "models" / "best.pt"

# Dimensiones de la webcam (mi web cam de logitech C920 es 1080p)
FRAME_W, FRAME_H = 1920, 1080  #

# Correccion del offset de Y
# La cámara está desplazada 5cm respecto al cañón.
# A 3 metros de distancia, 5cm aprox 24 píxeles
# Como la camara esta encima del cañon, restamos offset para compersar
OFFSET_Y_PX = -24 # (+ px -> camara debajo del cañon, - px -> camara arriba del cañon)

# 2. Clases de Hardware y Visión
Aquí he definido las dos clases principales que uso para cargar los modelos y controlar el hardware:
*   **TurretDriver**: Se encarga de enviar las señales al Arduino para mover los servos y disparar.
*   **VisionSystem**: Carga el la CNN (YOLO) entrenada y devuelve dónde está la diana en cada foto.

In [ ]:
class TurretDriver:
    """Controlador de interfaz Serial para la torreta."""
    def __init__(self, port, baud):
        self.ser = None
        if port is not None:
            self.ser = serial.Serial(port, baud, timeout=1)
            time.sleep(2)
            print(f"Puerto abierto: {port}")
        
        self.yaw = 0.0      
        self.pitch = 100.0  

    def send_action(self, action_idx):
        STEP = 2.0
        
        cmd = None
        
        if action_idx == 0:   # Pitch+ (Down)
            cmd = b'w'; self.pitch += STEP 
        elif action_idx == 1: # Pitch- (Up)
            cmd = b's'; self.pitch -= STEP 
        elif action_idx == 2: # Yaw- (Left)
            cmd = b'a'; self.yaw -= STEP   
        elif action_idx == 3: # Yaw+ (Right)
            cmd = b'd'; self.yaw += STEP   
        elif action_idx == 4: # Fire
            cmd = b'f'                     
        
        self.yaw = np.clip(self.yaw, -90, 90)
        self.pitch = np.clip(self.pitch, 33, 150)
        if cmd and self.ser is not None:
            self.ser.write(cmd)

    def close(self):
        if self.ser: self.ser.close()

class VisionSystem:
    """Wrapper para el modelo de detección de objetos."""
    def __init__(self, model_path):
        print("Cargando YOLO...")
        if not model_path.is_file():
            raise FileNotFoundError(f"No se encuentra el checkpoint YOLO: {model_path}")
        self.model = YOLO(str(model_path))
    
    def detect_target(self, frame):
        
        results = self.model(frame, verbose=False)
        
        best_box = None
        max_conf = 0.5
        
        for r in results:
            for box in r.boxes:
                if box.conf > max_conf:
                    max_conf = box.conf
                    best_box = box.xywh[0].cpu().numpy() 
        
        return best_box

## Prueba con una imagen

Ejecuta esta celda para comprobar el detector con una imagen de ejemplo del conjunto Shooting Target. Para los modos de cámara y hardware, ajusta los parámetros de arriba.

In [ ]:
from IPython.display import display
from PIL import Image

vision = VisionSystem(MODEL_YOLO_PATH)
demo_path = PROJECT_ROOT / "data" / "demo" / "target.jpg"
result = vision.model.predict(str(demo_path), device="cpu", verbose=False)[0]
display(Image.fromarray(cv2.cvtColor(result.plot(), cv2.COLOR_BGR2RGB)))

# 3. Ejecución del Sistema Autónomo

Este bloque implementa el flujo principal de control, dividido en tres fases:
1.  **Adquisición**: Detección y fijación del objetivo en el flujo de video.
2.  **Calibración**: Cálculo de coordenadas de referencia relativas.
3.  **Control**: Bucle de realimentación donde el agente de RL toma decisiones basándose en el estado estimado.


In [ ]:
# Correspondecia entre píxeles y grados
PX_PER_DEG_YAW = 27.0
PX_PER_DEG_PITCH = 25.0

# Inicialización de la torre y el sistema de visión
vision = VisionSystem(MODEL_YOLO_PATH)
agent = DQN.load(MODEL_RL_PATH, device="cpu")
turret = TurretDriver(SERIAL_PORT, BAUD_RATE)

cap = cv2.VideoCapture(CAMERA_INDEX) 

cap.set(cv2.CAP_PROP_FRAME_WIDTH, FRAME_W)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, FRAME_H)

if not cap.isOpened():
    turret.close()
    cap.release()
    raise RuntimeError("Cámara no encontrada.")


# 1. Encontrar el objetivo

print("Esperando adquisición del objetivo...")
frame_ref = None
target_data = None

while True:
    ret, frame = cap.read()
    if not ret: break
    
    # Procesamiento de baja resolución para velocidad
    frame_lite = cv2.resize(frame, (640, 480))
    target_data = vision.detect_target(frame_lite)
    
    if target_data is not None:
        print("Objetivo adquirido. Bloqueando frame de referencia.")
        frame_ref = frame 
        
        l, t, w, h = target_data
        cv2.rectangle(frame_lite, (int(l-w/2), int(t-h/2)), (int(l+w/2), int(t+h/2)), (0, 255, 0), 3)
        cv2.putText(frame_lite, "LOCKED", (20, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (0,255,0), 3)
        cv2.imshow("System Status", frame_lite)
        cv2.waitKey(500) 
        break 
    else:
        cv2.putText(frame_lite, "Buscando objetivo...", (20, 50), cv2.FONT_HERSHEY_SIMPLEX, 1, (0,0,255), 2)
        cv2.imshow("System Status", frame_lite)
        
    if cv2.waitKey(1) & 0xFF == ord('q'):
        print("Cancelado por el usuario.")
        break

cv2.destroyWindow("System Status")

if frame_ref is None:
    print("No se ha encontrado objetivo.")
    turret.close()
    cap.release()
    raise RuntimeError("No se ha encontrado objetivo.")

frame_ref = cv2.resize(frame_ref, (FRAME_W, FRAME_H))


# 2. Cálculos iniciales


scale_x = FRAME_W / 640.0
scale_y = FRAME_H / 480.0

tx_lite, ty_lite, tw_lite, th_lite = target_data

t_cx = tx_lite * scale_x
t_cy = ty_lite * scale_y
t_w  = tw_lite * scale_x
t_h  = th_lite * scale_y

print(f"Coordenadas del objetivo: {t_cx:.0f}, {t_cy:.0f}")

adjusted_t_cy = t_cy + OFFSET_Y_PX
init_err_x_px = t_cx - FRAME_W/2

print(f"Lanzando el auto-aim")


# 3. Ejecución del agente sobre la torreta

step = 0
max_steps = 50 

# Normalización de la imagen
cx_img, cy_img = FRAME_W // 2, FRAME_H // 2

while step < max_steps:
    ret_live, frame_live = cap.read()
    if ret_live:
        debug_frame = cv2.resize(frame_live, (FRAME_W, FRAME_H))
    else:
        debug_frame = frame_ref.copy()

    # Estimación del estado del modelo
    sim_ptr_x = cx_img + (turret.yaw * PX_PER_DEG_YAW)
    sim_ptr_y = cy_img - ((turret.pitch - 100.0) * PX_PER_DEG_PITCH)
    
    # Cálculo del error
    current_err_x_px = t_cx - sim_ptr_x
    current_err_y_px = sim_ptr_y - adjusted_t_cy 
    
    # Normalización
    norm_err_x = np.clip(current_err_x_px / (FRAME_W/2), -1.0, 1.0)
    norm_err_y = np.clip(current_err_y_px / (FRAME_H/2), -1.0, 1.0)
    norm_yaw = np.clip(turret.yaw / 90.0, -1.0, 1.0)
    norm_pitch = np.clip((turret.pitch - 100.0) / 60.0, -1.0, 1.0)
    
    obs = np.array([norm_err_x, norm_err_y, norm_yaw, norm_pitch], dtype=np.float32)
    
    # Inferencia
    action, _ = agent.predict(obs, deterministic=True)
    action = int(action.item())
    
    # Actuación
    turret.send_action(action)
    time.sleep(0.05)
    
    # Visualización de depuración
    pt_x, pt_y = int(sim_ptr_x), int(sim_ptr_y)
    pt_x = np.clip(pt_x, 0, FRAME_W-1); pt_y = np.clip(pt_y, 0, FRAME_H-1)
    
    cv2.drawMarker(debug_frame, (pt_x, pt_y), (255, 0, 0), markerType=cv2.MARKER_CROSS, markerSize=40, thickness=3)
    
    l, t_rect = int(t_cx - t_w/2), int(t_cy - t_h/2)
    r, b_rect = int(t_cx + t_w/2), int(t_cy + t_h/2)
    cv2.rectangle(debug_frame, (l, t_rect), (r, b_rect), (0, 255, 0), 2)
    
    cv2.line(debug_frame, (pt_x, pt_y), (int(t_cx), int(t_cy)), (0, 255, 255), 2)
    
    cv2.putText(debug_frame, f"STEP: {step}", (50, 60), cv2.FONT_HERSHEY_SIMPLEX, 1, (0,0,255), 2)
    action_names = ["DOWN", "UP", "LEFT", "RIGHT", "FIRE"]
    txt = action_names[action] if action < 5 else "?"
    cv2.putText(debug_frame, f"ACTION: {txt}", (50, 110), cv2.FONT_HERSHEY_SIMPLEX, 1.5, (0,0,255), 3)
    
    preview = cv2.resize(debug_frame, (960, 540)) 
    cv2.imshow("Turret View", preview)
    cv2.waitKey(100) 
    
    if action == 4:
        status = "Comando de disparo enviado" if turret.ser else "Acción simulada"
        print(status)
        
        start_obs = time.time()
        while time.time() - start_obs < 5.0:
            ret_live, frame_live = cap.read()
            if not ret_live: break
            
            cv2.putText(frame_live, f"{status}. Pulsa q para salir.", (50, 100), cv2.FONT_HERSHEY_SIMPLEX, 2, (0, 0, 255), 3)
            
            preview_end = cv2.resize(frame_live, (960, 540))
            cv2.imshow("Turret View", preview_end)
            
            if cv2.waitKey(1) & 0xFF == ord('q'): break
        break
        
    step += 1

cv2.waitKey(0)
cv2.destroyAllWindows()
turret.close()
cap.release()

# 4. Ejemplos de ejecución

- Sin tener la torreta es complicado ejecutar las funciones y comprobar que funcionan, por ello he grabado un par de ejemplos de uso y los he subido a Youtube:
> - Disparo 1: https://youtu.be/mGippGd279k
> - Disparo 2: https://youtu.be/KS9mdomQKo0
> - Disparo 3: https://youtu.be/VWXIKj431Yc


- Sin embargo si se ejecuta el bloque del punto 3 sin la torreta se puede ver el proceso que haría (localización y comandos de la torreta), lo único que los comandos seriales no se enviarán a ningun sitio.


# 5. Conclusiones

El sistema funciona correctamente y logra acertar en el objetivo, validando el diseño planteado. No obstante, he notado que en la realidad es un poco menos preciso que en la simulación teórica debido a dos factores que he identificado durante las pruebas:

1.  **Imperfecciones de los servos**: Aunque funcionan bien, he visto que a veces giran con distinta fuerza hacia un lado que hacia otro (asimetría), lo que introduce pequeñas desviaciones.
2.  **Resistencia del aire**: El modelo físico asume condiciones ideales. No he podido incluir el rozamiento del aire en las ecuaciones por falta de conocimientos avanzados, lo que afecta ligeramente a la trayectoria real de la bala.

## Funciones de debugging

In [ ]:
# Debug - Vista previa de la cámara con detección de objetivo

# Inicializar el modelo de visión y la captura de video
vision = VisionSystem(MODEL_YOLO_PATH)
cap = cv2.VideoCapture(CAMERA_INDEX)

if not cap.isOpened():
    print("Error al abrir la cámara")
else:
    print("Vista previa (Pulsa 'q' para salir)")
    while True:
        # Leer el frame actual de la cámara
        ret, frame = cap.read()
        if not ret:
            break

        # Redimensionar la imagen y realizar la detección
        frame = cv2.resize(frame, (640, 480))
        target = vision.detect_target(frame)

        if target is not None:
            # Calcular las coordenadas del cuadro delimitador (bounding box)
            cx, cy, w, h = target
            l, t = int(cx - w/2), int(cy - h/2)
            r, b = int(cx + w/2), int(cy + h/2)
            
            # Dibujar el rectángulo y la etiqueta del objetivo detectado
            cv2.rectangle(frame, (l, t), (r, b), (0, 255, 0), 2)
            cv2.putText(frame, "Target", (l, t-10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)
        else:
            # Mostrar mensaje de búsqueda si no se encuentra el objetivo
            cv2.putText(frame, "Searching", (20, 20), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)

        # Mostrar la ventana de previsualización
        cv2.imshow("Preview", frame)
        
        # Detener el bucle si se presiona la tecla 'q'
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    # Liberar la cámara y cerrar las ventanas
    cap.release()
    cv2.destroyAllWindows()

In [ ]:
# Debug - Prueba de movimiento

import time

turret = TurretDriver(SERIAL_PORT, BAUD_RATE)
time.sleep(2) 

test_sequence = [
    (1, "ARRIBA"),
    (1, "ARRIBA"),
    (0, "ABAJO"),
    (0, "ABAJO"),
    (2, "IZQUIERDA"),
    (2, "IZQUIERDA"),
    (3, "DERECHA"),
    (3, "DERECHA")
]

for action_id, name in test_sequence:
    turret.send_action(action_id)
    
    if turret.ser and turret.ser.in_waiting:
        try:
            turret.ser.readline()
        except:
            pass
            
    time.sleep(1) 

turret.close()

### Nota sobre la implementación
El procesamiento de imágenes mediante OpenCV (`cv2`) y la lógica de visualización en tiempo real los he hecho con asistencia de herramientas de IA generativa dado mi desconocimiento de OpenCV

## Código del Arduino

Consulta [el sketch de la entrega](../firmware/turret/turret.ino), que conserva su aviso MIT.